# Assignment 17.1 — Comparing Classifiers
**Bank Marketing Dataset | UCI Machine Learning Repository**

## Business Problem
A Portuguese bank ran a series of phone-based marketing campaigns trying to get clients to subscribe to a term deposit. The goal here is to predict whether a given client will say **yes** to the subscription based on a mix of their profile and the economic context at the time of the call.

This is a binary classification problem: `y = yes` (subscribed) or `y = no` (didn't subscribe).

I'll compare four classifiers — **KNN, Logistic Regression, Decision Tree, and SVM** — to see which performs best on this task.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, roc_auc_score,
    ConfusionMatrixDisplay, accuracy_score
)

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

## 2. Load & Explore Data

In [ ]:
df = pd.read_csv('bank-additional-full.csv', sep=';')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
# class balance — expecting it to be imbalanced
print(df['y'].value_counts())
print(f"\nSubscription rate: {(df['y']=='yes').mean()*100:.1f}%")

As expected, the dataset is imbalanced — only about 11% of calls result in a subscription. This is important to keep in mind when choosing an evaluation metric (accuracy alone would be misleading here).

## 3. Feature Selection

The dataset has 20 input features. I initially tried using all of them, but the models — especially SVM — were very slow and the results were harder to interpret. I narrowed it down to 6 features that cover each category of information in the dataset:

- **Client profile:** `age`, `job`
- **Call info:** `duration`, `campaign`
- **Economic context:** `euribor3m`, `emp.var.rate`

`duration` is worth noting — it's the length of the call in seconds. Longer calls probably mean the client was more engaged. It's a strong predictor but in a real deployment you wouldn't know the call duration before making the call, so I'll keep it but flag it.

In [ ]:
selected_features = ['age', 'job', 'duration', 'campaign', 'euribor3m', 'emp.var.rate']
target = 'y'

data = df[selected_features + [target]].copy()
data.head()

## 4. Data Cleaning & Preprocessing

In [ ]:
# check for missing values
print(data.isnull().sum())

In [ ]:
# job is categorical — encode it
# there are 'unknown' values in job, let's see how many
print(data['job'].value_counts())

In [ ]:
# 'unknown' is only 330 rows out of 41k — dropping them
data = data[data['job'] != 'unknown'].copy()
print(f'Rows after dropping unknown job: {len(data)}')

# encode job as dummy variables
data = pd.get_dummies(data, columns=['job'], drop_first=True)

# encode target: yes=1, no=0
data['y'] = (data['y'] == 'yes').astype(int)

print(f'Final shape: {data.shape}')
data.head(3)

## 5. Exploratory Data Analysis

In [ ]:
# subscription rate by job type
job_sub = df[df['job'] != 'unknown'].groupby('job')['y'].apply(lambda x: (x=='yes').mean()).sort_values(ascending=False)

plt.figure(figsize=(10, 4))
job_sub.plot(kind='bar', color='steelblue')
plt.ylabel('Subscription Rate')
plt.xlabel('Job Type')
plt.title('Term Deposit Subscription Rate by Job Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# call duration vs subscription — I expected this to be strong
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df[df['y']=='no']['duration'], bins=50, alpha=0.6, color='steelblue', label='No')
axes[0].hist(df[df['y']=='yes']['duration'], bins=50, alpha=0.6, color='salmon', label='Yes')
axes[0].set_xlabel('Call Duration (seconds)')
axes[0].set_ylabel('Count')
axes[0].set_title('Call Duration by Subscription Outcome')
axes[0].legend()
axes[0].set_xlim(0, 2000)  # trimming extreme outliers for readability

# euribor rate vs outcome
sns.boxplot(data=df, x='y', y='euribor3m', ax=axes[1], palette=['steelblue', 'salmon'])
axes[1].set_xticklabels(['No', 'Yes'])
axes[1].set_title('Euribor 3M Rate by Subscription Outcome')
axes[1].set_ylabel('Euribor 3M Rate')

plt.tight_layout()
plt.show()

In [ ]:
# interesting — euribor rate is noticeably lower for clients who subscribed
# makes sense: when rates are low, term deposits become more attractive

# age distribution
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x='age', hue='y', bins=40, kde=True, palette={'no': 'steelblue', 'yes': 'salmon'})
plt.title('Age Distribution by Subscription Outcome')
plt.xlabel('Age')
plt.tight_layout()
plt.show()

## 6. Train/Test Split & Scaling

In [ ]:
X = data.drop('y', axis=1)
y = data['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print(f'Train subscription rate: {y_train.mean():.1%} | Test: {y_test.mean():.1%}')

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

## 7. Modeling

I'll train all four classifiers and compare them. For evaluation I'm using **ROC-AUC** as the primary metric — because the dataset is imbalanced (~11% positive), accuracy would be misleading (a model that always predicts "no" gets 89% accuracy but is useless).

I'll also run cross-validation and GridSearchCV for each model.

### 7.1 Logistic Regression

In [ ]:
param_grid_lr = {'C': [0.01, 0.1, 1, 10]}
gs_lr = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42),
                     param_grid_lr, cv=5, scoring='roc_auc')
gs_lr.fit(X_train_sc, y_train)

best_lr = gs_lr.best_estimator_
lr_auc = roc_auc_score(y_test, best_lr.predict_proba(X_test_sc)[:, 1])
lr_acc = accuracy_score(y_test, best_lr.predict(X_test_sc))
print(f'Best C: {gs_lr.best_params_["C"]}')
print(f'Logistic Regression — Test AUC: {lr_auc:.4f} | Accuracy: {lr_acc:.4f}')

### 7.2 K-Nearest Neighbors

In [ ]:
param_grid_knn = {'n_neighbors': [3, 5, 9, 15, 21]}
gs_knn = GridSearchCV(KNeighborsClassifier(),
                      param_grid_knn, cv=5, scoring='roc_auc')
gs_knn.fit(X_train_sc, y_train)

best_knn = gs_knn.best_estimator_
knn_auc = roc_auc_score(y_test, best_knn.predict_proba(X_test_sc)[:, 1])
knn_acc = accuracy_score(y_test, best_knn.predict(X_test_sc))
print(f'Best k: {gs_knn.best_params_["n_neighbors"]}')
print(f'KNN — Test AUC: {knn_auc:.4f} | Accuracy: {knn_acc:.4f}')

### 7.3 Decision Tree

In [ ]:
param_grid_dt = {'max_depth': [3, 5, 8, None], 'min_samples_leaf': [5, 10, 20]}
gs_dt = GridSearchCV(DecisionTreeClassifier(random_state=42),
                     param_grid_dt, cv=5, scoring='roc_auc')
gs_dt.fit(X_train_sc, y_train)

best_dt = gs_dt.best_estimator_
dt_auc = roc_auc_score(y_test, best_dt.predict_proba(X_test_sc)[:, 1])
dt_acc = accuracy_score(y_test, best_dt.predict(X_test_sc))
print(f'Best params: {gs_dt.best_params_}')
print(f'Decision Tree — Test AUC: {dt_auc:.4f} | Accuracy: {dt_acc:.4f}')

### 7.4 Support Vector Machine

SVM on the full 40k dataset was taking way too long on my machine. I ended up using a random sample of 5,000 rows for training instead — still enough to get a meaningful result, but much faster.

In [ ]:
# SVM is slow on large datasets — subsample training set
sample_idx = np.random.RandomState(42).choice(len(X_train_sc), size=5000, replace=False)
X_train_svm = X_train_sc[sample_idx]
y_train_svm = y_train.iloc[sample_idx]

param_grid_svm = {'C': [0.1, 1, 10], 'kernel': ['rbf', 'linear']}
gs_svm = GridSearchCV(SVC(probability=True, random_state=42),
                      param_grid_svm, cv=3, scoring='roc_auc')  # cv=3 to keep it manageable
gs_svm.fit(X_train_svm, y_train_svm)

best_svm = gs_svm.best_estimator_
svm_auc = roc_auc_score(y_test, best_svm.predict_proba(X_test_sc)[:, 1])
svm_acc = accuracy_score(y_test, best_svm.predict(X_test_sc))
print(f'Best params: {gs_svm.best_params_}')
print(f'SVM — Test AUC: {svm_auc:.4f} | Accuracy: {svm_acc:.4f}')

## 8. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'KNN', 'Decision Tree', 'SVM'],
    'Test AUC': [lr_auc, knn_auc, dt_auc, svm_auc],
    'Test Accuracy': [lr_acc, knn_acc, dt_acc, svm_acc]
}).sort_values('Test AUC', ascending=False)

print(results.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['gold' if i == 0 else 'steelblue' for i in range(len(results))]

axes[0].barh(results['Model'], results['Test AUC'], color=colors)
axes[0].set_xlabel('ROC-AUC')
axes[0].set_title('Model Comparison — Test ROC-AUC')
axes[0].set_xlim(0.5, 1.0)
for i, (val, model) in enumerate(zip(results['Test AUC'], results['Model'])):
    axes[0].text(val + 0.002, i, f'{val:.3f}', va='center')

axes[1].barh(results['Model'], results['Test Accuracy'], color=colors)
axes[1].set_xlabel('Accuracy')
axes[1].set_title('Model Comparison — Test Accuracy')
axes[1].set_xlim(0.5, 1.0)
for i, val in enumerate(results['Test Accuracy']):
    axes[1].text(val + 0.002, i, f'{val:.3f}', va='center')

plt.suptitle('Note: Accuracy is misleading here due to class imbalance (~11% positive)',
             fontsize=9, color='gray')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves for all models
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(7, 6))

for model, name, X_input in [
    (best_lr, 'Logistic Regression', X_test_sc),
    (best_knn, 'KNN', X_test_sc),
    (best_dt, 'Decision Tree', X_test_sc),
    (best_svm, 'SVM', X_test_sc)
]:
    probs = model.predict_proba(X_input)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Classifiers')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# confusion matrix for the best model
best_model_name = results.iloc[0]['Model']
model_lookup = {
    'Logistic Regression': best_lr,
    'KNN': best_knn,
    'Decision Tree': best_dt,
    'SVM': best_svm
}
best_model = model_lookup[best_model_name]

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(best_model, X_test_sc, y_test,
                                       display_labels=['No', 'Yes'],
                                       cmap='Blues', ax=ax)
ax.set_title(f'Confusion Matrix — {best_model_name} (Test Set)')
plt.tight_layout()
plt.show()

## 9. Logistic Regression Coefficients

Since logistic regression performed competitively and is the most interpretable, it's worth looking at which features it weighted most heavily.

In [ ]:
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': best_lr.coef_[0]
}).sort_values('coefficient')

plt.figure(figsize=(9, 5))
colors = ['salmon' if c < 0 else 'steelblue' for c in coef_df['coefficient']]
plt.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Coefficient')
plt.title('Logistic Regression Coefficients')
plt.tight_layout()
plt.show()

## 10. Findings & Recommendations

### What the models found

**Call duration is the strongest predictor** — longer calls are strongly associated with a successful subscription. This makes intuitive sense: if someone stays on the phone longer, they were probably interested. However, this feature is only observable *after* the call, so it can't be used to decide *who* to call ahead of time. It's a useful signal for post-call analysis, but less useful for targeting.

**Economic context matters.** Lower euribor rates are associated with higher subscription rates — when market interest rates are low, locking in a term deposit becomes more attractive. This means the bank's conversion rate will naturally fluctuate with the economic cycle, somewhat independent of the campaign itself.

**Job type has some signal.** Retired clients and students tend to subscribe at higher rates than admin or blue-collar workers, possibly because they have different financial priorities or time availability.

### Model comparison summary

Logistic Regression and SVM came out on top in terms of AUC. Decision Tree was competitive but tends to overfit without careful depth tuning. KNN was the weakest performer — it struggles when features are on very different scales even after standardization.

Accuracy is misleading here (all models look great because 89% of the data is "no"). **AUC is the right metric** because it measures how well the model distinguishes between subscribers and non-subscribers across all thresholds.

### Recommendations (non-technical)

- **Time calls strategically** — campaigns run during low-interest-rate environments will naturally convert better. Consider timing pushes around favorable macro conditions.
- **Prioritize longer engagement** — training call center staff to keep clients on the line longer (through better scripts or more targeted pitches) correlates with higher conversion.
- **Segment by job type** — retired clients and students are more likely to subscribe. A campaign specifically targeting these segments could improve conversion rates without increasing call volume.

### Next steps
- Try removing `duration` from the feature set to build a model that can be used *before* a call is made — that would be more practically useful for targeting
- Explore resampling techniques (SMOTE, class weighting) to better handle the class imbalance